Limpieza



In [2]:
import re
import os

def limpiar_texto(ruta_archivo):
    if not os.path.exists(ruta_archivo):
        return "El archivo no existe."

    with open(ruta_archivo, 'r', encoding='utf-8') as f:
        texto = f.read()

    # 1. PRIMERO: Limpiar los espacios
    # Reemplaza múltiples espacios, saltos de línea o tabulaciones por un solo espacio y quita espacios en los bordes.
    texto_limpio = re.sub(r'(^ \w\s)', '', texto).strip()

    # 2. LUEGO: Quitar caracteres especiales y puntuación
    # Mantiene únicamente las letras (incluyendo acentos y ñ) y los espacios que acabamos de limpiar.
    texto_limpio = re.sub(r'[^a-zA-ZáéíóúñÁÉÍÓÚÑ\s]', '', texto_limpio)


    return texto_limpio

# Ejemplo con uno de los archivos listados
archivo_ejemplo = '/content/libro_1.txt'
resultado = limpiar_texto(archivo_ejemplo)

print(f"--- Texto original (primeros 100 caracteres) ---")
with open(archivo_ejemplo, 'r') as f: print(f.read()[:100])

print(f"\n--- Texto limpio (primeros 100 caracteres) ---")
print(resultado[:100])

--- Texto original (primeros 100 caracteres) ---
﻿The Project Gutenberg eBook of The Declaration of Independence of the United States of America

   

--- Texto limpio (primeros 100 caracteres) ---
The Project Gutenberg eBook of The Declaration of Independence of the United States of America

    


Tokenizacion

In [9]:
import nltk
import os
from nltk.tokenize import word_tokenize

# Descargamos los paquetes necesarios para separar las palabras (solo una vez)
nltk.download('punkt')
# ¡AQUÍ ESTÁ LA SOLUCIÓN! Agregamos esta línea:
nltk.download('punkt_tab')

ruta_archivo = 'libro_1.txt'

if os.path.exists(ruta_archivo):
    with open(ruta_archivo, 'r', encoding='utf-8-sig') as f:
        texto_libro = f.read()

        # Hacemos la tokenización: cortamos el texto en palabras
        mis_tokens = word_tokenize(texto_libro)

        print("--- RESULTADO DE LA TOKENIZACIÓN ---")
        # Imprimimos solo los primeros 20 tokens para no llenar la pantalla
        print(mis_tokens[:20])
else:
    print(f"No se encontró el archivo '{ruta_archivo}'.")

--- RESULTADO DE LA TOKENIZACIÓN ---
['The', 'Project', 'Gutenberg', 'eBook', 'of', 'The', 'Declaration', 'of', 'Independence', 'of', 'the', 'United', 'States', 'of', 'America', 'This', 'eBook', 'is', 'for', 'the']


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


Stemming

In [10]:
from nltk.stem import SnowballStemmer

# Inicializamos el "cortador de palabras" indicando que es para INGLÉS
stemmer = SnowballStemmer('english')

# Verificamos que la variable 'mis_tokens' (del Código 1) exista en la memoria
if 'mis_tokens' in locals():

    # Aplicamos el stemming: recorremos la lista y cortamos cada palabra
    mis_tokens_con_stemming = [stemmer.stem(palabra) for palabra in mis_tokens]

    print("--- RESULTADO DEL STEMMING ---")
    # Imprimimos los primeros 20 resultados para que veas el cambio
    print(mis_tokens_con_stemming[:20])

else:
    print("Error: No se encontró la lista 'mis_tokens'. Asegúrate de ejecutar la celda de Tokenización primero.")

--- RESULTADO DEL STEMMING ---
['the', 'project', 'gutenberg', 'ebook', 'of', 'the', 'declar', 'of', 'independ', 'of', 'the', 'unit', 'state', 'of', 'america', 'this', 'ebook', 'is', 'for', 'the']


Join

In [11]:
# Verificamos que la variable del paso anterior exista
if 'mis_tokens_con_stemming' in locals():

    # El método .join() toma la lista y une cada elemento usando lo que está entre comillas (un espacio ' ')
    texto_procesado_final = ' '.join(mis_tokens_con_stemming)

    print("--- RESULTADO DEL JOIN (TEXTO FINAL) ---")
    # Imprimimos los primeros 200 caracteres para ver cómo quedó el texto reconstruido
    print(texto_procesado_final[:200])

else:
    print("Error: No se encontró la lista 'mis_tokens_con_stemming'. Asegúrate de ejecutar la celda de Stemming primero.")

--- RESULTADO DEL JOIN (TEXTO FINAL) ---
the project gutenberg ebook of the declar of independ of the unit state of america this ebook is for the use of anyon anywher in the unit state and most other part of the world at no cost and with alm


TF-IDF

In [14]:
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd

# Verificamos que tienes el texto del paso anterior
if 'texto_procesado_final' in locals():

    # IMPORTANTE: TF-IDF está diseñado para comparar VARIOS documentos (un corpus).
    # Como ahorita solo tenemos un libro, lo metemos dentro de una lista para simular el corpus.
    # Si tuvieras más libros, sería algo así: [libro_1, libro_2, libro_3]
    corpus = [texto_procesado_final]

    # 1. Inicializamos la herramienta (el vectorizador)
    vectorizador = TfidfVectorizer()

    # 2. Matemáticas en acción: calculamos el TF-IDF
    matriz_tfidf = vectorizador.fit_transform(corpus)

    # 3. Extraemos las palabras que encontró y sus puntuaciones
    nombres_palabras = vectorizador.get_feature_names_out()
    puntuaciones = matriz_tfidf.toarray()[0] # [0] porque solo tenemos 1 documento

    # 4. Lo metemos en una tabla de Pandas para que sea fácil de leer
    df_tfidf = pd.DataFrame({
        'Palabra (Stem)': nombres_palabras,
        'Puntuación TF-IDF': puntuaciones
    })

    # Ordenamos de mayor a menor importancia
    df_ordenado = df_tfidf.sort_values(by='Puntuación TF-IDF', ascending=False)

    print("--- TOP 15 PALABRAS CON MAYOR PUNTUACIÓN TF-IDF ---")
    display(df_ordenado.head(15))

else:
    print("Error: No se encontró 'texto_procesado_final'. Asegúrate de ejecutar la celda del JOIN primero.")

--- TOP 15 PALABRAS CON MAYOR PUNTUACIÓN TF-IDF ---


,Palabra (Stem),Puntuación TF-IDF
854,the,0.573533
614,of,0.443783
876,to,0.293348
67,and,0.240696
404,gutenberg,0.189924
443,in,0.176761
708,project,0.171120
632,or,0.152315
972,work,0.146674
979,you,0.141033
